In [1]:
from segmentiq.builder import *
from segmentiq.scorer import *
from segmentiq.utils.data_loader import *

# Example Feature Groupings Setup
variable_groups = {
    "Demographics": [
        "age", 
        "gender", 
        "education_level", 
        "dependents", 
        "home_ownership", 
        "city_tier"
    ],
    "Employment": [
        "employment_type", 
        "years_employed"
    ],
    "Financial_Profile": [
        "annual_income", 
        "credit_score", 
        "existing_debt", 
        "account_balance", 
        "monthly_spend", 
        "loan_to_income_ratio", 
        "previous_defaults", 
        "number_of_cards"
    ],
    "Application_Details": [
        "requested_limit", 
        "application_channel"
    ]
}

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000,5000, 1000, 500, 100],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="is_approved",
    # n_jobs=-1,
    min_sample_size=1000,
    min_lift=1.1,
    min_events = 5,
    top_n_vars=15,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=1,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=variable_groups,
    ignore_features=['application_id'],
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/Strategic_Segment_Builder/package/src/credit_app_data.parquet").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-11 17:45:19,805 | INFO     | [builder.py:337] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-11 17:45:20,479 | INFO     | [builder.py:133] | Feature group validation passed. (18 features mapped)
2026-07-11 17:45:20,480 | INFO     | [builder.py:352] | Dynamic Grid Search Enabled: 10 total configurations.
2026-07-11 17:45:20,482 | INFO     | [builder.py:373] | Iteration 1 | Remaining Volume: 500,000 | Base Rate: 26.77%
2026-07-11 17:45:28,410 | INFO     | [builder.py:509] | Feature Usage Tracker Update -> 'loan_to_income_ratio' used count = 1
2026-07-11 17:45:28,411 | INFO     | [builder.py:524] | Segment 1 Captured (Size Floor: 10000 | Lift Floor: 2.0): loan_to_income_ratio < 0.07
2026-07-11 17:45:29,206 | INFO     | [builder.py:373] | Iteration 2 | Remaining Volume: 456,198 | Base Rate: 22.65%
2026-07-11 17:45:36,600 | INFO     | [builder.py:509] | Feature Usage Tracker Update -> 'credit_score' used count = 1
2026-07-11 17:45:36,600 | INFO     | [builder.py:524] | Se

In [2]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate | capture_rate |        lift        |
+---------+-------------+---------------+--------------------+--------------------+--------------+--------------------+
|   0.0   |   411613.0  |    79176.0    | 19.235544066878354 | 26.772000000000002 |   82.3226    | 0.7184948478588956 |
|   1.0   |   43802.0   |    30543.0    | 69.72969270809553  | 26.772000000000002 |    8.7604    | 2.6045754037089317 |
|   2.0   |   44585.0   |    24141.0    | 54.146013233150164 | 26.772000000000002 |    8.917     |  2.0224866738813   |
+---------+-------------+---------------+--------------------+--------------------+--------------+--------------------+


In [3]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   loan_to_income_ratio=(-inf, 0.07)
SQL Filter: loan_to_income_ratio < 0.07
--------------------------------------------------
Segment ID: 2
Raw Rule:   credit_score=[795.50, inf)
SQL Filter: credit_score >= 795.50
--------------------------------------------------


In [5]:
builder.explain_feature_journey("city_tier")

AUDIT TRAIL FOR FEATURE: 'city_tier'

[Iteration 1]
  • Current dynamic IV   : 0.0008
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : loan_to_income_ratio=(-inf, 0.07) (Variables: ['loan_to_income_ratio'])

[Iteration 2]
  • Current dynamic IV   : 0.0002
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : credit_score=[795.50, inf) (Variables: ['credit_score'])

[Iteration 3]
  • Current dynamic IV   : 0.0004
  • Previous times used  : 0
  • Selection Status     : Excluded (Outside Top N Features by IV)


In [11]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
            SELECT *, 
            CASE WHEN loan_to_income_ratio < 0.07
            THEN 1 ELSE 0 END AS seg_1,
            CASE WHEN credit_score >= 795.50
            THEN 1 ELSE 0 END AS seg_2,
            FROM predicted
""").df()
conn.close()

In [16]:
scorer = StrategicSegmentScore(
    target_col="is_approved",
    primary_key="application_id",
    segment_cols=["seg_1", "seg_2"],
)

In [17]:
scorer.calculate_and_export_weights(predicted)

2026-07-11 17:48:54,575 | INFO     | [scorer.py:50] | Initializing out-of-core DuckDB scorecard engine...
2026-07-11 17:48:55,598 | INFO     | [scorer.py:88] | Computing harmonic scorecard weights...
2026-07-11 17:48:55,599 | INFO     | [scorer.py:117] | Scoring 100M rows natively via C++ database engine...
2026-07-11 17:48:55,856 | INFO     | [scorer.py:132] | Dataset Zero-Inflation Rate: 73.23%
2026-07-11 17:48:55,856 | INFO     | [scorer.py:135] | Calibrating deciles across active populations...


{'model_metadata': {'total_training_population': 500000,
  'active_scored_population': 500000,
  'active_population_pct': 100.0,
  'baseline_event_rate': 0.2677},
 'segment_weights': {'seg_1': {'weight': 90,
   'lift': 2.6046,
   'response_rate': 0.6973,
   'capture_rate': 0.2282},
  'seg_2': {'weight': 67,
   'lift': 2.1684,
   'response_rate': 0.5805,
   'capture_rate': 0.2119}},
 'decile_min_thresholds': {'1': 157,
  '2': 67,
  '3': 0,
  '4': 0,
  '5': 0,
  '6': 0,
  '7': 0,
  '8': 0,
  '9': 0,
  '10': 0}}

In [18]:
scorecard_json_path = "/workspaces/Strategic_Segment_Builder/package/src/scorecard_model_2026_07_11_17_45_19.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-11 17:50:28,187 | INFO     | [3233594613.py:2] | Loading scorecard model artifact from /workspaces/Strategic_Segment_Builder/package/src/scorecard_model_2026_07_11_17_45_19.json...


In [19]:
model_artifact.get("decile_min_thresholds")

{'1': 157,
 '2': 67,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [20]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 90
Segment: seg_2 | Weight: 67


In [24]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 90 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 67 ELSE 0 END AS seg_2_weighted,
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=157 THEN 1
                    WHEN total_weight >= 67 THEN 2
                    WHEN total_weight >= 0 THEN 3
                    WHEN total_weight >= 0 THEN 4
                    WHEN total_weight >= 0 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [25]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(is_approved) AS events, 
                    (SUM(is_approved)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()
scored

,decile_band,count,events,response_rate
0,1,4282,4227.0,98.715553
1,2,84105,50457.0,59.992866
2,3,411613,79176.0,19.235544
